In [2]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [4]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [5]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [ ]:
# Read image data, feed into Claude
with open("images/prop7.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")
    
messages = []
add_user_message(messages, [
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_bytes
        }
    },
    {
        "type": "text",
        "text": prompt
    }
])

chat(messages)

Message(id='msg_01NEiQDnjzqbaM3fRZ75KeK6', container=None, content=[TextBlock(citations=None, text='# Satellite Image Fire Risk Analysis\n\n**1. Residence Identification:**\nThe primary residence is located in the center of the property as a light-colored, multi-section structure with an irregular footprint, surrounded on all sides by dense coniferous forest with minimal clearance to the property boundaries.\n\n**2. Tree Overhang Analysis:**\nExamination of the canopy reveals extensive tree coverage directly above the residence, with coniferous branches overhanging approximately 50-75% of the visible roof area, particularly concentrated on the northern and eastern sides of the structure where the densest tree growth occurs.\n\n**3. Fire Risk Assessment:**\nThe overhanging trees create multiple vulnerability points for wildfire risk, with continuous fuel pathways from the surrounding forest directly to the roof, and the dense canopy appears to create "bridge" connections between the wil

In [ ]:
# Read image data, feed into Claude
with open("images/prop3.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")
    
messages = []
add_user_message(messages, [
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_bytes
        }
    },
    {
        "type": "text",
        "text": prompt
    }
])

chat(messages)

Message(id='msg_01PxtBwEczBkqpPXVyewFxiR', container=None, content=[TextBlock(citations=None, text='# Satellite Property Analysis - Fire Risk Assessment\n\n**1. Residence Identification:**\nThe primary residence is the large multi-section structure with a dark roof (appearing to be two stories) located in the center-left portion of the property, with a connected white structure (likely a garage or addition) to its left, positioned approximately 50-75 feet from the tree lines on the north and east sides.\n\n**2. Tree Overhang Analysis:**\nModerate-density trees surround the property on all sides, with the most significant canopy coverage on the north (top) and east (right) sides; approximately 15-25% of the main residence roof appears to have direct tree canopy overhang, particularly affecting the upper (northern) section of the home.\n\n**3. Fire Risk Assessment:**\nThe overhanging branches create potential ember catch points and a bridge between the surrounding woodland vegetation and

In [ ]:
# Read document data, feed into Claude
with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")
    
messages = []
add_user_message(messages, [
    {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes
        }
    },
    {
        "type": "text",
        "text": "Summarize the contents of this document in 1 sentence."
    }
])

chat(messages)

Message(id='msg_01Eadm2r6uABnHPGKRzXguwc', container=None, content=[TextBlock(citations=None, text='# Summary\n\nThis Wikipedia article provides comprehensive information about Earth, covering its position as the third planet from the Sun, its unique characteristics as the only known planet harboring life, its physical and orbital properties, atmosphere, and natural history of formation and evolution.', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=9631, output_tokens=54, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

In [ ]:
# Read document data, feed into Claude
with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")
    
messages = []
add_user_message(messages, [
    {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes
        }
    },
    {
        "type": "text",
        "text": "How were Earth's ocean and atmosphere formed?"
    }
])

chat(messages)

Message(id='msg_01KJBh9gbQx76VioeU9smPKi', container=None, content=[TextBlock(citations=None, text="# Formation of Earth's Ocean and Atmosphere\n\nAccording to the document, Earth's atmosphere and oceans were formed through the following processes:\n\n## Atmospheric and Ocean Formation\n\n**Volcanic Activity and Outgassing**: Earth's atmosphere and oceans were formed by volcanic activity and outgassing.\n\n**Water Sources**: Water vapor from volcanic sources condensed into the oceans, and was augmented by water and ice from:\n- Asteroids\n- Protoplanets\n- Comets\n\nThe document notes that sufficient water to fill the oceans may have been present on Earth since its formation.\n\n## Early Atmospheric Conditions\n\n**Greenhouse Effect**: In the early Earth, atmospheric greenhouse gases played a crucial role by keeping the oceans from freezing when the newly forming Sun had only 70% of its current luminosity.\n\n## Magnetic Field Development\n\n**Atmospheric Protection**: By 3.5 billion y

In [10]:
# Read document data, feed into Claude
with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")
    
messages = []
add_user_message(messages, [
    {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes
        },
        "title": "earth.pdf",
        "citations": {"enabled": True}
    },
    {
        "type": "text",
        "text": "How were Earth's ocean and atmosphere formed?"
    }
])

chat(messages)

Message(id='msg_01RgFnJ5Ag1At2J8Jc3T4azt', container=None, content=[TextBlock(citations=[CitationPageLocation(cited_text="[42]\r\nEarth's atmosphere and oceans were formed by volcanic activity and outgassing.\r\n", document_index=0, document_title='earth.pdf', end_page_number=5, file_id=None, start_page_number=4, type='page_location')], text="Earth's atmosphere and oceans were formed by volcanic activity and outgassing.", type='text'), TextBlock(citations=None, text=' ', type='text'), TextBlock(citations=[CitationPageLocation(cited_text='[43] Water vapor from\r\nthese sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets,\r\nand comets.\r\n', document_index=0, document_title='earth.pdf', end_page_number=5, file_id=None, start_page_number=4, type='page_location')], text='Water vapor from these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets, and comets.', type='text'), TextBlock(citations=None, text=' ', typ

In [11]:
article_text = """
Earth
The Blue Marble
,
Apollo 17
, December 1972
Designations
Alternativenames
The
world ·
The
globe ·
Terra ·
Tellus ·
Gaia ·
Mother Earth ·
Sol III
Adjectives
Earthly ·
Terrestrial ·
Terran
·
Tellurian
Symbol
and
Orbital characteristics
Epoch
J2000
[
n 1
]
Aphelion
152
097
597
km
Perihelion
147
098
450
km
[
n 2
]
Semi-major axis
149
598
023
km
[
1
]
Eccentricity
0.016
7086
[
1
]
Orbital period(sidereal)
365.256
363
004
d
[
2
]
(
1.000
017
420
96
a
j
)
Average
orbitalspeed
29.7827 km/s
[
3
]
Mean anomaly
358.617°
Inclination
7.155°
–
Sun
's equator;
Earth
Earth
is the third
planet
from the
Sun
and the only
astronomical object
known to
harbor life
. This isenabled by Earth being an
ocean world
, the only one inthe
Solar System
sustaining liquid
surface water
. Almostall of Earth's water is contained in its global ocean,covering
70.8%
of
Earth's crust
. The remaining 29.2% ofEarth's crust is land, most of which is located in theform of
continental
landmasses
within Earth's
landhemisphere
. Most of Earth's land is at least somewhat
humid
and covered by vegetation, while large
sheets ofice
at
Earth's polar
deserts
retain more water thanEarth's
groundwater
, lakes, rivers, and
atmosphericwater
combined. Earth's crust consists of slowly moving
tectonic plates
, which interact to produce mountainranges,
volcanoes
, and earthquakes.
Earth has a liquidouter core
that generates a
magnetosphere
capable ofdeflecting most of the destructive
solar winds
and
cosmic radiation
.
Earth has
a dynamic atmosphere
, which sustainsEarth's surface conditions and protects it from most
meteoroids
and
UV-light at entry
. It has a compositionof primarily
nitrogen
and
oxygen
.
Water vapor
is widelypresent in the atmosphere,
forming clouds
that covermost of the planet. The water vapor acts as a
greenhousegas
and, together with other greenhouse gases in theatmosphere, particularly
carbon dioxide
(CO
2
), createsthe conditions for both liquid surface water and watervapor to persist via the capturing of
energy from theSun's light
. This process maintains the current averagesurface temperature of 14.76 °C (58.57 °F), at whichwater is liquid under normal atmospheric pressure.Differences in the amount of captured energy betweengeographic regions (as with the
equatorial region
receiving more sunlight than the polar regions) drive
atmospheric
and
ocean currents
, producing a global
climate system
with different
climate regions
, and arange of weather phenomena such as
precipitation
,allowing components such as
nitrogen
to
cycle
.
1.578
69
°
–
invariableplane
;
[
4
]
0.000
05
°
– J2000
ecliptic
Longitude ofascending node
−11.260
64
°
– J2000ecliptic
[
3
]
Time ofperihelion
2023-Jan-04
[
5
]
Argument ofperihelion
114.207
83
°
[
3
]
Satellites
1, the
Moon
Physical characteristics
Mean radius
6
371
.0 km
[
6
]
Equatorial
radius
6
378
.137 km
[
7
]
[
8
]
Polar
radius
6
356
.752 km
[
9
]
Flattening
1/
298.257
222
101
(
ETRS89
)
[
10
]
Circumference
40
075
.017 km
equatorial
[
8
]
40
007
.86 km
meridional
[
11
]
[
n 3
]
Surface area
510
072
000
km
2
[
12
]
[
n 4
]
Land:
148
940
000
km
2
Water:
361
132
000
km
2
Volume
1.083
21
×
10
12
km
3
[
3
]
Mass
5.972
168
×
10
24
kg
[
13
]
Mean
density
5.513 g/cm
3
[
3
]
Surface gravity
9.806
65
m/s
2
[
14
]
(exactly 1
g
0
)
Moment ofinertia factor
0.3307
[
15
]
Escape velocity
11.186 km/s
[
3
]
Synodic rotationperiod
1.0 d
(24h 00 m 00s)
Sidereal rotationperiod
0.997
269
68
d
[
16
]
(23h 56 m 4.100s)
Equatorialrotation velocity
0.4651 km/s
[
17
]
Axial tilt
23.439
2811
°
[
2
]
Albedo
0.434
geometric
[
3
]
0.294
Bond
[
3
]
Earth is
rounded
into
an ellipsoid
with
a circumference
of about 40,000 kilometres (25,000 miles). It is the
densest planet in the Solar System
. Of the four
rockyplanets
, it is the largest and most massive. Earth isabout eight
light-minutes
away from the Sun and
orbitsit
, taking a year (about 365.25 days) to complete onerevolution.
Earth rotates
around its own axis in slightlyless than a day (in about 23 hours and 56 minutes).
Earth's axis of rotation
is tilted with respect to theperpendicular to its orbital plane around the Sun,producing seasons. Earth is
orbited
by one permanent
natural satellite
, the
Moon
, which orbits Earth at384,400 km (238,900 mi)—1.28 light seconds—and isroughly a quarter as wide as Earth. The Moon's gravityhelps stabilize Earth's axis, causes
tides
and
graduallyslows Earth's rotation
.
Tidal locking
has made the Moonalways face Earth with the same side.
Earth, like most other bodies in the Solar System,
formed about 4.5 billion years ago
from gas and dust inthe
early Solar System
. During the first billion years of
Earth's history
, the ocean formed and then
lifedeveloped
within it. Life spread globally and has beenaltering Earth's atmosphere and surface, leading to the
Great Oxidation Event
two billion years ago. Humansemerged 300,000 years ago in Africa and have spreadacross every continent on Earth. Humans depend onEarth's
biosphere
and natural resources for theirsurvival, but have
increasingly impacted the planet'senvironment
. Humanity's current impact on Earth'sclimate and biosphere is unsustainable, threatening thelivelihood of humans and many other forms of life, and
causing widespread extinctions
.
[
23
]
The
Modern English
word
Earth
developed, via
MiddleEnglish
, from an
Old English
noun most often spelled
eorðe
.
[
24
]
It has cognates in every
Germanic language
,from which
Proto-Germanic
*
erþō
has beenreconstructed. In its earliest attestation, the word
eorðe
was used to translate the many senses of Latin
terra
andGreek
gē
: the ground, its soil, dry land, the humanworld, the surface of the world (including the sea), andthe globe itself. As with Roman
Terra
(or
Tellus
) and
Etymology
Temperature
255 K (−18 °C)
(
blackbody temperature
)
[
18
]
Surface
temp.
min
mean
max
[
n 5
]
−89.2 °C
14.76 °C
56.7 °C
Surface
equivalent dose
rate
0.274 μSv/h
[
22
]
Absolutemagnitude
(H)
−3.99
Atmosphere
Surface
pressure
101.325
kPa
(at sea level)
Composition byvolume
78.08%
nitrogen
(dry air)
20.95%
oxygen
(dry air)
≤1%
water vapor
(variable)
0.9340%
argon
0.0415%
carbon dioxide
0.00182%
neon
0.00052%
helium
0.00017%
methane
0.00011%
krypton
0.00006%
hydrogen
Source:
[
3
]
Greek
Gaia
, Earth may have been a
personified goddess
in
Germanic paganism
: late
Norse mythology
included
Jörð
('Earth'), a giantess often given as the mother of
Thor
.
[
25
]
Historically,
Earth
has been written in lowercase.During the
Early Middle English
period, its
definitesense
as "the globe" began being expressed using thephrase
the earth
. By the period of
Early ModernEnglish
,
capitalization of nouns began to prevail
, and
the earth
was also written
the Earth
, particularly whenreferenced along with other heavenly bodies. Morerecently, the name is sometimes simply given as
Earth
,by analogy with the names of the
other planets
, though
earth
and forms with
the earth
remain common.
[
24
]
House styles
now vary:
Oxford spelling
recognizes thelowercase form as the more common, with thecapitalized form an acceptable variant. Anotherconvention capitalizes
Earth
when appearing as a name,such as a description of the "Earth's atmosphere", butemploys the lowercase when it is preceded by
the
, suchas "the atmosphere of the earth". It almost alwaysappears in lowercase in colloquial expressions such as"what on earth are you doing?"
[
26
]
The name
Terra
/
ˈ
t
ɛr
ə
/
TERR
-ə
is occasionally used inscientific writing; it also sees use in science fiction todistinguish humanity's inhabited planet from others,
[
27
]
while in poetry
Tellus
/
ˈ
t
ɛ
l
ə
s
/
TELL
-əs
hasbeen used to denote personification of the Earth.
[
28
]
Terra
is also the name of the planet in some
Romance languages
, languages that evolved from Latin, like Italian and Portuguese, while in otherRomance languages the word gave rise to names with slightly altered spellings, like the Spanish
Tierra
and the French
Terre
. The Latinate form
Gaea
(
English:
/
ˈ
dʒ
iː
.
ə
/
DJEE
-ə
) of the Greek poetic name
Gaia
(
[ɡâi̯ i̯ .a]
or
[ɡâj.ja]
) is rare, though the alternative spelling
Gaia
has become common due to the
Gaia hypothesis
, in which case its pronunciation is
/
ˈ
ɡ
aɪ
.
ə
/
GYE
-ə
rather than the more traditionalEnglish
/
ˈ
ɡ
eɪ
.
ə
/
GAY
-ə
.
[
29
]
There are a number of adjectives for the planet Earth. The word
earthly
is derived from
Earth
. Fromthe Latin
Terra
comes
terran
/
ˈ
t
ɛr
ə
n
/
TERR
-ən
,
[
30
]
terrestrial
/
t
ə
ˈ
r
ɛ
s
t
r
i
ə
l
/
tərr-
EHST
-ree-əl
,
[
31
]
and(via French)
terrene
/
t
ə
ˈ
r
iː
n
/
tə-
REEN
,
[
32
]
and from the Latin
Tellus
comes
tellurian
/
t
ɛ
ˈ
l
ʊər
i
ə
n
/
teh-
LUURR
-ee-ən
[
33
]
and
telluric
.
[
34
]
A 2012 artistic impression of the early
Solar System
's
protoplanetary disk
from which Earth and other SolarSystem bodies were formed
The oldest material found in the
Solar System
isdated to
4.5682
+0.0002
−0.0004
Ga
(billion years) ago.
[
35
]
By
4.54
±
0.04 Ga
the primordial Earth hadformed.
[
36
]
The bodies in
the Solar System formedand evolved
with the Sun. In theory, a
solar nebula
partitions a volume out of a
molecular cloud
bygravitational collapse, which begins to spin andflatten into a
circumstellar disk
, and then theplanets grow out of that disk with the Sun. Anebula contains gas, ice grains, and
dust
(including
primordial nuclides
). According to
nebular theory
,
planetesimals
formed by
accretion
, with theprimordial Earth being estimated as likely takinganywhere from 70 to 100 million years to form.
[
37
]
Estimates of the age of the Moon range from 4.5Ga to significantly younger.
[
38
]
A
leading hypothesis
is that it was formed by accretion from materialloosed from Earth after a
Mars
-sized object with about 10% of Earth's mass, named
Theia
, collidedwith Earth.
[
39
]
It hit Earth with a glancing blow and some of its mass merged with Earth.
[
40
]
[
41
]
Between approximately 4.0 and
3.8 Ga
, numerous
asteroid impacts
during the
Late HeavyBombardment
caused significant changes to the greater surface environment of the Moon and, byinference, to that of Earth.
[
42
]
Earth's atmosphere
and oceans were formed by
volcanic activity
and
outgassing
.
[
43
]
Water vapor fromthese sources
condensed
into the oceans, augmented by water and ice from asteroids,
protoplanets
,and
comets
.
[
44
]
Sufficient water to fill the oceans may have been on Earth since it formed.
[
45
]
In thismodel, atmospheric
greenhouse gases
kept the oceans from freezing when the newly forming Sun
hadonly 70%
of its
current luminosity
.
[
46
]
By
3.5 Ga
,
Earth's magnetic field
was established, which helpedprevent the atmosphere from being stripped away by the
solar wind
.
[
47
]
As the molten outer layer of Earth cooled it
formed
the first solid
crust
, which is thought to have been
mafic
in composition. The first
continental crust
, which was more
felsic
in composition, formed by thepartial melting of this mafic crust.
[
49
]
The presence of grains of the
mineral zircon of Hadean age
in
Eoarchean
sedimentary rocks
suggests that at least some felsic crust existed as early as
4.4 Ga
, only
140
Ma
after Earth's formation.
[
50
]
There are two main models of how this initial small volume ofcontinental crust evolved to reach its current abundance:
[
51
]
(1) a relatively steady growth up to thepresent day,
[
52
]
which is supported by the radiometric dating of continental crust globally and (2) aninitial rapid growth in the volume of continental crust during the
Archean
, forming the bulk of the
Natural history
Formation
After formation
"""

In [12]:
# Read document data, feed into Claude
with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")
    
messages = []
add_user_message(messages, [
    {
        "type": "document",
        "source": {
            "type": "text",
            "media_type": "text/plain",
            "data": article_text,
        },
        "title": "Earth Article",
        "citations": {"enabled": True}
    },
    {
        "type": "text",
        "text": "How were Earth's ocean and atmosphere formed?"
    }
])

chat(messages)

Message(id='msg_01BB22FbKheEhTTnhxszfD5T', container=None, content=[TextBlock(citations=[CitationCharLocation(cited_text="[\n42\n]\nEarth's atmosphere\nand oceans were formed by\nvolcanic activity\nand\noutgassing\n.\n", document_index=0, document_title='Earth Article', end_char_index=10164, file_id=None, start_char_index=10077, type='char_location')], text="Earth's atmosphere and oceans were formed by volcanic activity and outgassing.", type='text'), TextBlock(citations=None, text=' ', type='text'), TextBlock(citations=[CitationCharLocation(cited_text='[\n43\n]\nWater vapor fromthese sources\ncondensed\ninto the oceans, augmented by water and ice from asteroids,\nprotoplanets\n,and\ncomets\n.\n', document_index=0, document_title='Earth Article', end_char_index=10298, file_id=None, start_char_index=10164, type='char_location')], text='Water vapor from these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets, and comets.', type='text')], model='cl